# Module 14: Intruder Alert!

## Mission Briefing

Last module, your Pico learned to measure distance with sound. Today it learns to detect **motion** — like a security system that knows when someone walks into a room.

The secret? Everything warm gives off invisible **infrared light**. A PIR sensor can detect when something warm moves past it — and it doesn't need to send out any signal at all. It just watches.

```
                   +-----------+
                   |  PIR      |
                   | Sensor    |
                   |   (O)     |  <-- Fresnel lens
                   +-----+-----+
                         |
     Person walks by --> | <-- detects IR change
                         |
                   +-----+-----+
                   |  Pico 2W  |
                   +-----------+
                    |    |    |
                  Buzzer LED Button
                  (alarm)(warning)(arm/disarm)
```

By the end of today, you'll have a working security system. Time to protect your desk!

---
## Science Spotlight: Passive Infrared Detection

Everything warm gives off invisible infrared light — including you! A PIR sensor can detect when something warm moves past it.

The "P" in PIR stands for **Passive** — the sensor doesn't send out any signal. It just quietly watches for changes in infrared light. When a warm body (like a person) moves across its field of view, the sensor notices the change and sends a signal.

*Your instructor will explain how PIR sensors work, what infrared light is, and introduce a powerful new concept: interrupts.*

---
## Meet the HC-SR501 PIR Sensor

The HC-SR501 is a motion detector commonly used in security lights and automatic room lights. Under the white dome (called a **Fresnel lens**), there are two tiny infrared-sensitive elements.

```
    +-----------------------+
    |      Fresnel Lens     |
    |        (  O  )        |  <-- white dome
    +-----------------------+
    |                       |
    |   [sensitivity]       |  <-- orange knob (adjust range)
    |   [delay time ]       |  <-- orange knob (adjust hold time)
    |                       |
    +---+-------+-------+--+
        |       |       |
       VCC     OUT     GND
        |       |       |
       5V     GP14    GND
```

- **VCC** = power (5V)
- **OUT** = output signal — goes HIGH when motion is detected
- **GND** = ground
- **Two adjustment knobs** on the board:
  - Sensitivity: how sensitive to movement (turn clockwise = more sensitive)
  - Delay time: how long OUT stays HIGH after detection (turn clockwise = longer)

**Important:** The PIR sensor needs **30-60 seconds to warm up** when first powered on. During this time it may give false readings. Be patient!

---
## Wiring: PIR Sensor + Buzzer + LED + Button

### Circuit

```
   PIR Sensor:
   Pico VBUS (5V, pin 40) ──── wire ──── PIR VCC
   Pico GND  (pin 38)     ──── wire ──── PIR GND
   Pico GP14 (pin 19)     ──── wire ──── PIR OUT

   Buzzer:
   Pico GP15 (pin 20)     ──── wire ──── Buzzer (+) ──── Buzzer (-) ──── GND

   LED:
   Pico GP13 (pin 17)     ──── [220 ohm] ──── LED (+) ──── LED (-) ──── GND

   Button (for arming the system):
   Pico GP12 (pin 16)     ──── wire ──── Button ──── GND
```

### Step-by-Step

1. **Place the PIR sensor** — it has 3 pins. Check the labels on the board (pin order varies by manufacturer!)
2. **Connect a wire** from **VBUS (5V)** on the Pico to **VCC** on the PIR
3. **Connect a wire** from **GND** on the Pico to **GND** on the PIR
4. **Connect a wire** from **GP14** on the Pico to **OUT** on the PIR
5. **Wire the buzzer** from **GP15** to **GND** (positive leg to GP15)
6. **Wire the LED** from **GP13** through a **220-ohm resistor** to **GND**
7. **Wire the button** from **GP12** to **GND** (we'll use the internal pull-up resistor)

**Important:** After plugging in USB, **wait 30-60 seconds** before testing. The PIR sensor needs time to calibrate. Stay still during this time!

Plug in your USB cable and wait patiently.

---
## Code Along: Reading the PIR Sensor

The PIR sensor output is simple: it's a **digital signal**.
- **HIGH (1)** = motion detected
- **LOW (0)** = no motion

Let's just read it and print the result. Remember — wait for the sensor to warm up first!

Fill in the blanks:

In [ ]:
from machine import Pin
import time

pir = Pin(_____, Pin.IN)        # Which GPIO pin is the PIR OUT connected to?

print("Warming up sensor... stay still!")
time.sleep(3)                    # Brief wait (sensor may need more on first power-up)
print("Ready! Try waving your hand.")

while True:
    if pir._____() == 1:         # How do we read a digital input pin?
        print("Motion detected!")
    else:
        print("All clear.")
    time.sleep(0.5)

Wave your hand in front of the sensor. You should see "Motion detected!" appear. When you stop moving, it should go back to "All clear" (after the delay time set by the sensor's knob).

**Troubleshooting:** If you see constant "Motion detected!" even when nobody is moving, the sensor is still warming up. Unplug, wait 5 seconds, plug back in, and stay completely still for 60 seconds.

---
## Code Along: Motion Alarm with LED and Buzzer

Now let's make a real alarm! When motion is detected:
1. Turn on the LED
2. Sound the buzzer for 3 seconds
3. Turn everything off and go back to watching

Fill in the blanks:

In [ ]:
from machine import Pin, PWM
import time

pir = Pin(14, Pin.IN)
led = Pin(_____, Pin.OUT)        # Which GPIO pin is the LED on?
buzzer = PWM(Pin(_____))         # Which GPIO pin is the buzzer on?
buzzer.freq(2000)
buzzer.duty_u16(0)               # Start with buzzer off

print("Warming up... stay still!")
time.sleep(3)
print("Motion alarm ACTIVE!")

while True:
    if pir.value() == _____:     # What value means "motion detected"?
        print("*** INTRUDER DETECTED! ***")
        led.value(1)             # LED on
        buzzer.duty_u16(5000)    # Buzzer on
        time.sleep(_____)        # Alarm for how many seconds?
        led.value(0)             # LED off
        buzzer.duty_u16(0)       # Buzzer off
        print("Alarm reset. Watching...")
    time.sleep(0.1)

Walk past the sensor — the LED should light up and the buzzer should sound for 3 seconds, then reset.

Notice how the code is constantly checking `pir.value()` in a `while True` loop. This is called **polling** — like constantly looking out the window to see if someone is at the door. It works, but the Pico can't do anything else while it's checking. There's a better way...

---
## Code Along: Interrupts — A Smarter Way to Watch

Instead of constantly checking the sensor in a loop (**polling**), we can tell the Pico: "Hey, interrupt whatever you're doing when the PIR detects motion." This is called an **interrupt**.

Think of it this way:
- **Polling** = constantly checking if someone is at the door
- **Interrupt** = installing a doorbell — you go about your business until it rings

Fill in the blanks:

In [ ]:
from machine import Pin, PWM
import time

pir = Pin(14, Pin.IN)
led = Pin(13, Pin.OUT)
buzzer = PWM(Pin(15))
buzzer.freq(2000)
buzzer.duty_u16(0)

motion_detected = False       # A flag to track if motion happened

def motion_handler(pin):      # This function runs when motion is detected
    global motion_detected
    motion_detected = _____   # Set the flag — what value means "yes, motion happened"?

# Set up the interrupt
pir.irq(trigger=Pin.IRQ_RISING, handler=_____)    # Which function should run?

print("Warming up... stay still!")
time.sleep(3)
print("Interrupt-based alarm ACTIVE!")
print("(The Pico is free to do other things while waiting...)")

count = 0
while True:
    if motion_detected:
        print("*** INTRUDER! ***")
        led.value(1)
        buzzer.duty_u16(5000)
        time.sleep(3)
        led.value(0)
        buzzer.duty_u16(0)
        motion_detected = False    # Reset the flag
        print("Alarm reset.")
    
    # The Pico can do other work here!
    count += 1
    if count % 20 == 0:            # Every ~2 seconds
        print("Still watching... (loop count:", count, ")")
    time.sleep(0.1)

Notice something different? The main loop is doing its own thing (counting), and the PIR interrupt fires **only when motion happens**. The Pico doesn't waste time constantly reading the sensor.

**Key concept:** `Pin.IRQ_RISING` means "interrupt me when the pin goes from LOW to HIGH" — which is exactly when the PIR detects motion.

The `motion_handler` function is called a **callback** — it's a function that gets "called back" by the hardware when something happens.

---
## Experiments

Try these and write down what you observe:

1. **Detection range:** Stand at different distances from the sensor and walk past it. How far away can it detect you? (Typical: 3-7 meters)

2. **Detection angle:** Stand to the side of the sensor and walk past. How wide is its field of view? (Typical: about 110 degrees)

3. **Warm vs cold:** Wave your hand in front of the sensor. Now wave a cold water bottle. Does it detect the bottle? Why or why not?

4. **Sensitivity adjustment:** Carefully turn the sensitivity knob on the PIR sensor (use a small screwdriver). How does it change the detection range?

5. **Delay time:** Turn the delay knob. How does it change how long the OUT pin stays HIGH after motion?

---
## Challenge: Full Security System

Build a complete security system with these features:

1. **Arm/Disarm button** — Press the button to arm or disarm the system
2. **Status LED** — Blinks slowly when armed, off when disarmed
3. **Motion trigger** — When armed AND motion is detected, trigger the alarm
4. **10-second alarm** — LED flashes rapidly and buzzer sounds for 10 seconds
5. **Auto-rearm** — After the alarm, the system goes back to armed mode

### Hints:
- You'll need a variable like `armed = False` to track the system state
- Use the button from Module 4 with `Pin.PULL_UP` (pressed = 0)
- For the flashing alarm, use a `for` loop inside the alarm code
- Remember to **debounce** the button (add a short `time.sleep(0.3)` after detecting a press)

```
   System States:
   
   DISARMED ──(button press)──> ARMED
       ^                          |
       |                    (motion detected)
  (button press)                  |
       |                          v
   ARMED <──(10 sec timeout)── ALARM!
```

Get creative! Ask your instructor if you get stuck.

---
## Vocabulary Recap

| Word | What It Means |
|------|---------------|
| **PIR** | Passive Infrared — a sensor that detects moving warm objects by their infrared radiation |
| **Infrared (IR)** | Invisible light given off by warm objects — just below visible red light |
| **Passive** | The sensor doesn't send out any signal — it only watches (unlike ultrasonic which sends pulses) |
| **Fresnel lens** | The white dome on the PIR sensor that focuses infrared light onto the detector |
| **Polling** | Constantly checking a sensor in a loop — like looking out the window over and over |
| **Interrupt** | Telling the Pico to stop and run a function when something happens — like a doorbell |
| **IRQ_RISING** | Trigger the interrupt when a pin goes from LOW to HIGH |
| **Callback** | A function that gets called automatically when an event (like an interrupt) happens |

---
*Circuit Explorers — Module 14: Intruder Alert!*